# DLinear Multi-Horizon Training — Google Colab

**Purpose:** Train DLinear separately for each prediction horizon (48h, 72h, 192h, 336h)

**Why:** Demonstrate that DLinear requires retraining per horizon, unlike D2Vformer's flexible forecasting.

**Datasets:** ETTh1 and Delhi AQI

**Note:** 96h checkpoints already exist from Semester 1, we only train the missing horizons.

---

## Training Plan

| Dataset | Horizons to Train | Expected Time |
|---------|------------------|---------------|
| ETTh1 | 48h, 72h, 192h, 336h | ~40 min |
| IndiaAQI | 48h, 72h, 192h, 336h | ~30 min |
| **Total** | **8 training runs** | **~70 min** |

---

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Navigate to Project Directory

**Update the path below to match your Google Drive structure**

In [ ]:
%cd /content/drive/MyDrive/MajorProject/D2Vformer/D2Vformer

## Step 3: Check GPU Availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Training will be slow.")
    print("Go to Runtime > Change runtime type > GPU")

## Step 4: Install Dependencies (if needed)

In [ ]:
# Uncomment if any packages are missing
# !pip install pandas numpy matplotlib seaborn tqdm pyyaml -q

## Step 5: Verify Dataset Paths

In [ ]:
import os

datasets_to_check = [
    './datasets/ETT-small/ETTh1.csv',
    './datasets/ETT-small/china.csv',
    './datasets/india_aqi/delhi_aqi.csv',
    './datasets/india_aqi/delhi_mark.csv'
]

print("Checking dataset files...")
all_exist = True
for path in datasets_to_check:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    print(f"{status} {path}")
    if not exists:
        all_exist = False

if all_exist:
    print("\n✓ All datasets found!")
else:
    print("\n✗ Some datasets missing. Upload them to Google Drive before continuing.")

## Step 6: Define Training Function

In [ ]:
import subprocess
import time

def train_dlinear(data_name, pred_len, epochs=50, patience=5):
    """
    Train DLinear for specific horizon.
    """
    print(f"\n{'='*70}")
    print(f"Starting: DLinear {data_name} pred_len={pred_len}h")
    print(f"{'='*70}")
    
    cmd = [
        'python', 'baselines/run_dlinear_multihorizon.py',
        '--data_name', data_name,
        '--pred_len', str(pred_len),
        '--epochs', str(epochs),
        '--patience', str(patience),
        '--batch_size', '64',
        '--lr', '0.001'
    ]
    
    start = time.time()
    result = subprocess.run(cmd, capture_output=False, text=True)
    elapsed = time.time() - start
    
    if result.returncode == 0:
        print(f"\n✓ SUCCESS: {data_name} pred_len={pred_len}h completed in {elapsed:.1f}s")
        return True
    else:
        print(f"\n✗ FAILED: {data_name} pred_len={pred_len}h")
        return False

## Step 7: Train All Horizons — ETTh1

**Training:** 48h, 72h, 192h, 336h  
**Skipping:** 96h (already trained in Semester 1)

In [ ]:
etth1_horizons = [48, 72, 192, 336]
etth1_results = []

for pred_len in etth1_horizons:
    success = train_dlinear('ETTh1', pred_len, epochs=50, patience=5)
    etth1_results.append((pred_len, success))

print("\n" + "="*70)
print("ETTh1 Training Summary")
print("="*70)
for pred_len, success in etth1_results:
    status = "✓ PASS" if success else "✗ FAIL"
    print(f"{pred_len}h: {status}")
print("="*70)

## Step 8: Train All Horizons — Delhi AQI

**Training:** 48h, 72h, 192h, 336h  
**Skipping:** 96h (already trained in Semester 1)

In [ ]:
aqi_horizons = [48, 72, 192, 336]
aqi_results = []

for pred_len in aqi_horizons:
    success = train_dlinear('IndiaAQI', pred_len, epochs=50, patience=5)
    aqi_results.append((pred_len, success))

print("\n" + "="*70)
print("Delhi AQI Training Summary")
print("="*70)
for pred_len, success in aqi_results:
    status = "✓ PASS" if success else "✗ FAIL"
    print(f"{pred_len}h: {status}")
print("="*70)

## Step 9: Consolidate Results

In [ ]:
import json
import os

# Load all individual result files
all_results = []

for data_name in ['ETTh1', 'IndiaAQI']:
    for pred_len in [48, 72, 192, 336]:
        result_file = f'experiments_flexible/dlinear_{data_name}_pred{pred_len}_result.json'
        if os.path.exists(result_file):
            with open(result_file, 'r') as f:
                result = json.load(f)
                all_results.append(result)
                print(f"Loaded: {result_file}")
        else:
            print(f"Missing: {result_file}")

# Save consolidated results
consolidated_file = 'experiments_flexible/dlinear_all_horizons.json'
with open(consolidated_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\nConsolidated results saved to: {consolidated_file}")
print(f"Total results: {len(all_results)}")

## Step 10: List Generated Checkpoints

In [ ]:
import glob

checkpoints = glob.glob('./baselines/dlinear_*.pkl')
checkpoints.sort()

print("\nGenerated DLinear Checkpoints:")
print("="*70)
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / (1024 * 1024)
    print(f"{os.path.basename(ckpt):<40} {size_mb:>6.2f} MB")
print("="*70)
print(f"Total checkpoints: {len(checkpoints)}")

## Step 11: Download Checkpoints to Local

**Option A: Download via Colab UI**
- Navigate to Files panel
- Find `baselines/` folder
- Download each checkpoint manually

**Option B: Create a ZIP archive**

In [ ]:
import shutil

# Create ZIP of all new checkpoints and results
shutil.make_archive('dlinear_multihorizon_results', 'zip', '.', 
                    base_dir=None)

# Alternative: Create ZIP of just checkpoints
!zip -q dlinear_checkpoints.zip baselines/dlinear_*.pkl

# Alternative: Create ZIP of just results
!zip -q dlinear_results.zip experiments_flexible/dlinear_*.json

print("\nZIP archives created:")
print("  dlinear_checkpoints.zip — all checkpoint files")
print("  dlinear_results.zip — all result JSON files")
print("\nDownload them from the Files panel on the left.")

## Step 12: Training Complete

**Next steps (on local machine):**

1. Download checkpoints and results from Colab
2. Copy to local `D2Vformer/D2Vformer/baselines/` and `experiments_flexible/`
3. Run comparison script to generate final results table
4. Create plots (MSE vs horizon, training cost comparison)
5. Update documentation

---

**Training Summary:**

- **DLinear training runs:** 8 (4 horizons × 2 datasets)
- **D2Vformer training runs (from Semester 1):** 2 (1 per dataset)
- **Retraining advantage:** D2Vformer requires 4× fewer training runs per dataset

This demonstrates D2Vformer's flexible forecasting capability.